In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "LINKUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,is_trending,hour,hour_sin,hour_cos,dow_sin,dow_cos,dom_sin,dom_cos,month_sin,month_cos
0,2025-09-01 00:00:00+00:00,23.20,23.20,23.16,23.19,5790.70,2025-09-01 00:00:59.999999+00:00,134263.0539,306,3676.64,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16
1,2025-09-01 00:01:00+00:00,23.19,23.21,23.18,23.21,1085.32,2025-09-01 00:01:59.999999+00:00,25186.7380,87,267.24,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16
2,2025-09-01 00:02:00+00:00,23.20,23.21,23.14,23.16,4998.64,2025-09-01 00:02:59.999999+00:00,115773.2040,305,2216.97,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16
3,2025-09-01 00:03:00+00:00,23.16,23.18,23.15,23.17,7742.80,2025-09-01 00:03:59.999999+00:00,179288.2392,201,2439.87,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16
4,2025-09-01 00:04:00+00:00,23.16,23.16,23.08,23.09,13156.35,2025-09-01 00:04:59.999999+00:00,304131.7741,551,3107.48,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 273,577
[info] optuna train rows: 175,088
[info] valid rows:        43,773
[info] test rows:         54,716


In [9]:
pruner = MedianPruner(n_warmup_steps=5, n_min_trials=10)
study = optuna.create_study(direction="maximize", pruner=pruner)

class EarlyStoppingCallback:
    def __init__(self, patience: int):
        self.patience = patience
        self.best_value = -float('inf')
        self.no_improvement_count = 0

    def __call__(self, study, trial):
        if study.best_value > self.best_value:
            self.best_value = study.best_value
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1

        if self.no_improvement_count >= self.patience:
            study.stop()

early_stopping = EarlyStoppingCallback(patience=10)

objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, callbacks=[early_stopping], show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-20 07:05:20,470] A new study created in memory with name: no-name-b39b8b17-a1b2-46fb-95b5-f2a710b3650d


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:06<?, ?it/s]

Best trial: 0. Best value: 0.0738689:   0%|          | 0/50 [00:06<?, ?it/s]

Best trial: 0. Best value: 0.0738689:   2%|▏         | 1/50 [00:06<05:01,  6.16s/it]

...

Best trial: 13. Best value: 0.0882111:  46%|████▌     | 23/50 [01:25<02:09,  4.81s/it]

Best trial: 13. Best value: 0.0882111:  46%|████▌     | 23/50 [01:25<02:09,  4.81s/it]

Best trial: 13. Best value: 0.0882111:  48%|████▊     | 24/50 [01:25<01:56,  4.50s/it]

Best trial: 13. Best value: 0.0882111:  48%|████▊     | 24/50 [01:25<01:32,  3.57s/it]

[I 2026-03-20 07:06:46,191] Trial 23 finished with value: 0.08475422601822849 and parameters: {'n_estimators': 1200, 'max_depth': 7, 'learning_rate': 0.0013143752789165923, 'subsample': 0.8350595846340388, 'colsample_bytree': 0.832797616823662, 'min_child_weight': 18, 'reg_alpha': 0.0002402669980592994, 'reg_lambda': 0.0001547157069089511}. Best is trial 13 with value: 0.08821111084291221.

[optuna] best trial
value: 0.088211
params:
  n_estimators: 800
  max_depth: 9
  learning_rate: 0.0011049651597353986
  subsample: 0.8561391383692846
  colsample_bytree: 0.7561100846163216
  min_child_weight: 20
  reg_alpha: 0.031333023567383274
  reg_lambda: 2.7964864187664444e-06


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final xgb...


[training] done in 4.98s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...



===== RESULTS =====
Train IC:      0.558923
Test IC:       0.059398
Train Rank IC: 0.166165
Test Rank IC:  0.098174
Train RMSE:    0.003344
Test RMSE:     0.002816


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
trend_strength      0.166665
volume_z            0.104797
volume_mom_5        0.085606
imbalance_15        0.077854
dom_sin             0.063606
dist_ma_30          0.055468
dist_ma_15_z        0.044045
range_ratio         0.040126
is_trending         0.034450
vol_30              0.028952
mom_5               0.026215
dow_cos             0.024181
imbalance_5         0.021851
vol_ratio_5_30      0.019469
hour_sin            0.019089
vol_15              0.018465
dist_ma_15          0.018353
vol_regime_ratio    0.015744
mom_3               0.014784
bar_range           0.014524
dist_ma_5           0.014249
range_15            0.014178
hour_cos            0.013276
vol_5               0.012937
mom_10              0.010489
mom_15              0.008616
range_5             0.007787
month_sin           0.006978
dow_sin             0.006114
dom_cos             0.005940
month_cos           0.005191
dtype: float32


In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/LINKUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/LINKUSDT__h5_model.joblib
[saved] features -> models/xgb/LINKUSDT__h5_feature_cols.json
[saved] feature importance -> models/xgb/LINKUSDT__h5_feature_importance.csv
[saved] metadata -> models/xgb/LINKUSDT__h5_meta.json
